# Energy-rank & calibrated-rank stability analysis

Для каждого чекпоинта каждого запуска считаем и сравниваем со своим финальным чекпоинтом через Spearman ρ:

- `energy_rank_95`, `energy_rank_99` — эффективный ранг по SVD ΔW при порогах энергии 95/99% (method-agnostic).
- `delta_w_rel` — ‖ΔW‖_F / ‖W‖_F (magnitude, method-agnostic).
- `gate_mean` — средняя магнитуда гейта |c_i| (только L1RA, непрерывно).
- `active_rank_cal16`, `active_rank_cal32` — counts `(|c_i| > τ)` с τ, подобранным бинарным поиском так, чтобы среднее по модулям = 16 или 32 (L1RA, калиброванный бюджет).

Контраст с `delta_w_rel`:
- `energy_rank_τ` — форма спектра, инвариантен к глобальному масштабу ΔW.
- `active_rank_cal@K` — прямой сигнал для per-module ранг-паттерна при фиксированном бюджете K. Решает проблему: L1RA с λ=1e-3 имеет 'natural' active_rank ≈ 28–30 при τ=0.1, что делает прямое сравнение с AdaLoRA target_r=16 некорректным.
- `gate_mean` — без дискретизации, не страдает от шума вблизи порога.

### Как запускать
1. В Colab подмонтируй Drive.
2. Убедись, что `PROJECT_DIR` указывает на папку с `logs_v2/`.
3. Run All. Результаты — в `analysis_v2/energy_rank_*.csv` и `analysis_v2/energy_rank_stab_curves.png`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip -q install safetensors scipy

In [ ]:
import os, re, json, time
from collections import defaultdict
import numpy as np
import pandas as pd
import torch
from scipy.stats import spearmanr

PROJECT_DIR = '/content/drive/MyDrive/diploma/project'
LOGS_DIR    = f'{PROJECT_DIR}/logs_v2'
OUT_DIR     = f'{PROJECT_DIR}/analysis_v2'
os.makedirs(OUT_DIR, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE  = torch.float32

ENERGY_THRESHOLDS = [0.95, 0.99]
L1RA_GATE_THR_NAIVE = 0.1        # для сравнения с нашим прежним analyze_v2
L1RA_CAL_TARGETS = [8, 16]      # бюджеты для бинарного поиска threshold
ONLY_DATASETS = ['meta_math', 'open_orca']   # None = все; ['meta_math','open_orca'] когда отбегают оба
print('Device:', DEVICE, '| ONLY_DATASETS:', ONLY_DATASETS)


## 1. Loaders для чекпоинтов

In [ ]:
from safetensors.torch import load_file

# Pattern: base_model.model.<path>.lora_{A|B|E}.default.weight  (adalora/lora)
#          base_model.model.<path>.lora_{A|B|c}_default          (l1ra .pt)
LORA_KEY_RE = re.compile(r'(?:base_model\.model\.|model\.)(?P<path>.+?)\.lora_(?P<role>[ABEc])(?:[._]|$)')
TARGET_MODULES = ['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj']

def _parse_key(k: str):
    m = LORA_KEY_RE.search(k)
    if not m: return None, None, None
    path = m.group('path')
    layer_m = re.search(r'layers\.(\d+)\.', path)
    mod_m   = re.search(r'\.(q_proj|k_proj|v_proj|o_proj|gate_proj|up_proj|down_proj)(\.|$)', path)
    if layer_m is None or mod_m is None: return None, None, None
    return int(layer_m.group(1)), mod_m.group(1), m.group('role')

def load_checkpoint(ckpt_dir: str):
    st = os.path.join(ckpt_dir, 'adapter_model.safetensors')
    pt = os.path.join(ckpt_dir, 'adapter_model.pt')
    if os.path.isfile(st):
        raw = load_file(st, device='cpu')
    elif os.path.isfile(pt):
        raw = torch.load(pt, map_location='cpu')
        if isinstance(raw, dict) and 'state_dict' in raw:
            raw = raw['state_dict']
    else:
        return {}
    out = defaultdict(dict)
    for k, v in raw.items():
        li, mn, role = _parse_key(k)
        if li is None: continue
        role_key = {'A':'A','B':'B','E':'E','c':'C'}[role]
        out[(li, mn)][role_key] = v.to(dtype=DTYPE, device=DEVICE, non_blocking=True)
    return dict(out)


## 2. Метрики

σ(ΔW) через QR двух факторов (rank-space SVD): если ΔW = L·R, то σ(ΔW) = σ(R₁·R₂ᵀ), где R₁,R₂ — малые (r×r) из reduced QR. Avoids O(mn·min(m,n)) SVD.

In [ ]:
def _svdvals_lowrank(left: torch.Tensor, right: torch.Tensor) -> torch.Tensor:
    _, R1 = torch.linalg.qr(left,    mode='reduced')
    _, R2 = torch.linalg.qr(right.T, mode='reduced')
    return torch.linalg.svdvals(R1 @ R2.T)

def module_sigma(tensors: dict, method: str) -> torch.Tensor:
    A = tensors.get('A'); B = tensors.get('B')
    if A is None or B is None: return None
    if method == 'lora':
        return _svdvals_lowrank(B, A)
    if method == 'adalora':
        E = tensors.get('E')
        if E is not None:
            return _svdvals_lowrank(B * E.flatten().view(1, -1), A)
        return _svdvals_lowrank(B, A)
    if method == 'l1ra':
        C = tensors.get('C')
        if C is not None:
            return _svdvals_lowrank(A * C.view(1, -1), B)
        return _svdvals_lowrank(A, B)
    raise ValueError(method)

def energy_rank(sigma: torch.Tensor, threshold: float) -> int:
    s2 = sigma ** 2
    total = float(s2.sum())
    if total <= 0: return 0
    s2_sorted, _ = torch.sort(s2, descending=True)
    cum = torch.cumsum(s2_sorted, dim=0) / total
    return int((cum >= threshold).nonzero(as_tuple=False)[0].item()) + 1

def delta_w_rel(tensors: dict, method: str) -> float:
    """‖ΔW‖_F / r, где r — ранг (прокси нормализации, безразмерно относительно самой ΔW).
    NB: это упрощение — в analyze_v2 делили на ‖W_base‖_F. Тут для стабильности ρ внутри одного рана
    это не важно: любое общее нормирование сокращается в Spearman."""
    A = tensors.get('A'); B = tensors.get('B')
    if A is None or B is None: return float('nan')
    sigma = module_sigma(tensors, method)
    return float((sigma ** 2).sum().sqrt())

## 3. L1RA: калиброванный active_rank при фиксированном бюджете

Для каждого чекпоинта собираем все |c_i| по всем модулям. Бинарным поиском находим **один общий** порог τ такой, что среднее по модулям `(|c_i| > τ).sum() ≈ K`. Используем для K=16, 32.

Это ровно `find_threshold_for_target_active_rank` из твоего `rank_analysis.ipynb`.

In [ ]:
def find_l1ra_threshold(gate_magnitudes_per_module: dict, target_avg: float,
                        n_iters: int = 30) -> float:
    """Бинарный поиск порога τ >= 0 такой, что mean_over_modules[(|c| > τ).sum()] ≈ target_avg.
    Полностью векторизовано на GPU: пэддим все гейты в матрицу (N_modules, r_max),
    недостающие позиции заполняем -inf (не проходят порог). .item() только в самом конце."""
    tensors = [g.flatten() for g in gate_magnitudes_per_module.values()]
    if not tensors:
        return 0.0
    dev = tensors[0].device
    r_max = max(t.numel() for t in tensors)
    padded = torch.full((len(tensors), r_max), float('-inf'),
                        device=dev, dtype=tensors[0].dtype)
    for i, t in enumerate(tensors):
        padded[i, :t.numel()] = t
    finite = padded.masked_fill(torch.isinf(padded), 0.0)
    lo = torch.zeros((), device=dev)
    hi = finite.max()
    target = torch.tensor(target_avg, device=dev, dtype=padded.dtype)
    for _ in range(n_iters):
        mid = 0.5 * (lo + hi)
        counts = (padded > mid).sum(dim=1).to(padded.dtype)
        avg = counts.mean()
        # без синка с CPU — выбор ветки бинарного поиска через where
        lo = torch.where(avg > target, mid, lo)
        hi = torch.where(avg > target, hi,  mid)
    return float((0.5 * (lo + hi)).item())

def l1ra_gates(tensors_by_module: dict) -> dict:
    """Возвращает {(layer, module): |c|} для всех модулей с ключом C."""
    out = {}
    for key, t in tensors_by_module.items():
        C = t.get('C')
        if C is not None:
            out[key] = C.abs().flatten()
    return out

## 4. Проходим все чекпоинты

In [ ]:
def infer_method(run_name: str) -> str:
    if 'adalora' in run_name: return 'adalora'
    if 'l1ra'    in run_name: return 'l1ra'
    return 'lora'

def infer_dataset(run_name: str) -> str:
    for d in ['meta_math','open_code_instruct','open_orca']:
        if run_name.startswith(d):
            return d
    return 'unknown'

rows = []
runs = sorted([r for r in os.listdir(LOGS_DIR) if os.path.isdir(f'{LOGS_DIR}/{r}')])
if ONLY_DATASETS:
    runs = [r for r in runs if any(r.startswith(d) for d in ONLY_DATASETS)]
print('Runs:', runs)

for run in runs:
    method  = infer_method(run)
    dataset = infer_dataset(run)
    ckpt_root = f'{LOGS_DIR}/{run}/checkpoints'
    if not os.path.isdir(ckpt_root):
        print(f'skip (no checkpoints/): {run}'); continue
    ckpts = sorted(
        [d for d in os.listdir(ckpt_root) if d.startswith('step_')],
        key=lambda x: int(x.split('_')[1])
    )
    t0 = time.time()
    for ckpt in ckpts:
        step = int(ckpt.split('_')[1])
        tensors = load_checkpoint(f'{ckpt_root}/{ckpt}')
        if not tensors: continue

        # Для L1RA — собираем гейты и калибруем пороги на этом чекпоинте
        cal_thresholds = {}
        if method == 'l1ra':
            gates = l1ra_gates(tensors)
            for K in L1RA_CAL_TARGETS:
                cal_thresholds[K] = find_l1ra_threshold(gates, target_avg=K)

        for (li, mn), t in tensors.items():
            sigma = module_sigma(t, method)
            if sigma is None: continue
            row = {
                'run_name': run, 'method': method, 'dataset': dataset,
                'step': step, 'layer': li, 'module': mn,
                'delta_w_rel': delta_w_rel(t, method),
            }
            for thr in ENERGY_THRESHOLDS:
                row[f'energy_rank_{int(thr*100)}'] = energy_rank(sigma, thr)

            # L1RA-specific
            C = t.get('C')
            if method == 'l1ra' and C is not None:
                c_abs = C.abs().flatten()
                row['gate_mean'] = float(c_abs.mean())
                row['active_rank_naive10'] = int((c_abs > L1RA_GATE_THR_NAIVE).sum().item())
                for K, thr in cal_thresholds.items():
                    row[f'active_rank_cal{K}'] = int((c_abs > thr).sum().item())
                    row[f'cal{K}_threshold']   = float(thr)
            else:
                row['gate_mean'] = float('nan')
                row['active_rank_naive10'] = int('nan') if False else float('nan')
                for K in L1RA_CAL_TARGETS:
                    row[f'active_rank_cal{K}'] = float('nan')
                    row[f'cal{K}_threshold']   = float('nan')
            rows.append(row)
    print(f'  {run}  ({len(ckpts)} ckpt, {time.time()-t0:.1f}s)')

per_ckpt = pd.DataFrame(rows)
per_ckpt.to_csv(f'{OUT_DIR}/energy_rank_per_checkpoint.csv', index=False)
print('Saved', f'{OUT_DIR}/energy_rank_per_checkpoint.csv', 'rows=', len(per_ckpt))
per_ckpt.head()


## 5. Санити-чек: какие threshold получились для L1RA?

Ожидаем, что τ для target=16 будет больше, чем для target=32 (более жёсткое отсечение). Если τ почти не меняется со step — значит распределение `|c|` стационарно.

In [ ]:
l1ra = per_ckpt[per_ckpt['method'] == 'l1ra']
if not l1ra.empty:
    thr_summary = l1ra.groupby(['run_name','step']).agg(
        cal8_thr=('cal8_threshold', 'first'),
        cal16_thr=('cal16_threshold', 'first'),
        active_cal8_mean=('active_rank_cal8', 'mean'),
        active_cal16_mean=('active_rank_cal16', 'mean'),
        active_naive10_mean=('active_rank_naive10', 'mean'),
    ).reset_index()
    thr_summary.to_csv(f'{OUT_DIR}/l1ra_thresholds_over_time.csv', index=False)
    print(thr_summary.to_string(index=False))
else:
    print('L1RA run не найден')


## 6. Spearman ρ(step, final) по всем метрикам

Внутри одного рана: на каждом `step` собираем вектор метрики по всем (layer, module), сравниваем со вектором на `final_step`.

In [ ]:
METRICS = [
    'delta_w_rel',
    'energy_rank_95',
    'energy_rank_99',
    'gate_mean',
    'active_rank_naive10',
    'active_rank_cal8',
    'active_rank_cal16',
]

stab_rows = []
for (run, method, dataset), g in per_ckpt.groupby(['run_name','method','dataset']):
    final_step = g['step'].max()
    g_final = g[g['step'] == final_step].set_index(['layer','module'])
    for step, gs in g.groupby('step'):
        gs = gs.set_index(['layer','module'])
        common = g_final.index.intersection(gs.index)
        if len(common) < 5: continue
        for m in METRICS:
            v_now   = gs.loc[common, m].values
            v_final = g_final.loc[common, m].values
            # пропускаем NaN-метрики (gate_mean у AdaLoRA, например)
            if np.all(np.isnan(v_now)) or np.all(np.isnan(v_final)):
                continue
            mask = ~np.isnan(v_now) & ~np.isnan(v_final)
            if mask.sum() < 5: continue
            a, b = v_now[mask], v_final[mask]
            if np.std(a) == 0 or np.std(b) == 0:
                rho = np.nan
            else:
                rho, _ = spearmanr(a, b)
            stab_rows.append({
                'run_name': run, 'method': method, 'dataset': dataset,
                'metric': m, 'step': step, 'final_step': final_step,
                'rel_progress': step / final_step,
                'rho_vs_final': rho,
            })

stab = pd.DataFrame(stab_rows)
stab.to_csv(f'{OUT_DIR}/energy_rank_stab_spearman.csv', index=False)
print('Saved', f'{OUT_DIR}/energy_rank_stab_spearman.csv  rows=', len(stab))
stab.head()


## 7. Точки стабилизации: min step где ρ ≥ τ (и держится до конца)

In [ ]:
def first_stable_step(g: pd.DataFrame, threshold: float):
    g = g.sort_values('step')
    above = g['rho_vs_final'] >= threshold
    if not above.any(): return None
    idx = np.where(above.values)[0]
    for i in idx:
        if above.values[i:].all():
            return int(g['step'].values[i])
    return None

rows = []
for (run, method, dataset, metric), g in stab.groupby(['run_name','method','dataset','metric']):
    final_step = int(g['final_step'].iloc[0])
    for thr in [0.8, 0.9, 0.95]:
        s = first_stable_step(g, thr)
        rows.append({
            'run_name': run, 'method': method, 'dataset': dataset,
            'metric': metric, 'threshold': thr,
            'stable_step': s,
            'rel_stable': (s/final_step) if s is not None else None,
        })

stab_pts = pd.DataFrame(rows)
stab_pts.to_csv(f'{OUT_DIR}/energy_rank_stabilization_points.csv', index=False)
print('Saved', f'{OUT_DIR}/energy_rank_stabilization_points.csv')
print('\n-- ρ ≥ 0.9 --')
print(stab_pts[stab_pts['threshold']==0.9]\
        .sort_values(['method','dataset','metric'])\
        .to_string(index=False))

## 8. Графики: ρ(step) по метрикам, усреднённые по датасетам внутри метода

In [ ]:
import matplotlib.pyplot as plt

agg = stab.groupby(['method','metric','step'])['rho_vs_final'].mean().reset_index()

# По методу — панель, по метрике — кривая внутри
methods = sorted(agg['method'].unique())
fig, axes = plt.subplots(1, len(methods), figsize=(6*len(methods), 4), sharey=True)
if len(methods) == 1: axes = [axes]

for ax, method in zip(axes, methods):
    sub = agg[agg['method'] == method]
    for metric in METRICS:
        g = sub[sub['metric'] == metric]
        if g.empty or g['rho_vs_final'].isna().all():
            continue
        ax.plot(g['step'], g['rho_vs_final'], marker='o', ms=2, lw=1.2, label=metric)
    ax.axhline(0.9, ls='--', color='gray', lw=0.8)
    ax.set_title(method)
    ax.set_xlabel('step')
    ax.grid(alpha=0.3)
    ax.legend(fontsize=7)
axes[0].set_ylabel('ρ vs final')
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/energy_rank_stab_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved', f'{OUT_DIR}/energy_rank_stab_curves.png')

## 9. Сводная: где метрика пересекает ρ=0.9 (в шагах и в % обучения)

In [ ]:
piv = stab_pts[stab_pts['threshold']==0.9]\
        .pivot_table(index=['method','dataset'], columns='metric', values='stable_step')
print('Step at which ρ crosses 0.9 and stays:')
print(piv.to_string())

piv_rel = stab_pts[stab_pts['threshold']==0.9]\
        .pivot_table(index=['method','dataset'], columns='metric', values='rel_stable')
print('\nAs a fraction of training:')
print((piv_rel.round(3)).to_string())

piv.to_csv(f'{OUT_DIR}/stabilization_step_pivot.csv')
piv_rel.to_csv(f'{OUT_DIR}/stabilization_relprogress_pivot.csv')

## 10. Как читать результаты

- **`delta_w_rel`** — magnitude signal. Ранжирование стабилизируется рано, но уязвимо к 'freeze artifact' (ρ→1 просто от затухания градиента).
- **`energy_rank_95/99`** — shape signal, инвариант к глобальному масштабу. Если стабилизируется на том же шаге, что delta_w_rel → структурный сигнал реален.
- **`gate_mean`** (L1RA) — непрерывный сигнал, без пороговой дискретизации. Чистая проверка стабильности важности.
- **`active_rank_cal16/32`** (L1RA) — бюджетно-откалиброванная метрика. Это та, которую напрямую возьмём в эксперимент B как per-module rank_pattern.
- **`active_rank_naive10`** (L1RA) — baseline из analyze_v2, оставлен для полноты.

**Главный вопрос, на который отвечает ноутбук:**

Совпадают ли шаги стабилизации для magnitude-метрик (`delta_w_rel`, `gate_mean`) и shape-метрик (`energy_rank_95`, `active_rank_cal16`)?

Если да — структура действительно фиксируется рано, и мы спокойно берём `active_rank_cal16(step=500)` как static rank_pattern для эксперимента B.

Если shape-метрики стабилизируются позже magnitude — значит ранжирование `delta_w_rel` ловит только затухание нормы, и заявлять про раннюю диагностику нельзя.

## 8. Overlay: loss vs rank-stability

Главный график для защиты — показывает: структура (rank) фиксируется **раньше**, чем лосс перестаёт падать.

Читаем `metrics.csv` каждого прогона (per-step train_loss при `log_per_step=True`) и overlay-им с ρ-кривыми из `stab`.


In [ ]:
def load_run_metrics(run_name: str) -> pd.DataFrame:
    p = f'{LOGS_DIR}/{run_name}/metrics.csv'
    if not os.path.isfile(p):
        return pd.DataFrame()
    df = pd.read_csv(p)
    df['run_name'] = run_name
    df['method']   = infer_method(run_name)
    df['dataset']  = infer_dataset(run_name)
    return df

runs_all = sorted([r for r in os.listdir(LOGS_DIR) if os.path.isdir(f'{LOGS_DIR}/{r}')])
if ONLY_DATASETS:
    runs_all = [r for r in runs_all if any(r.startswith(d) for d in ONLY_DATASETS)]

all_metrics = pd.concat([load_run_metrics(r) for r in runs_all], ignore_index=True)
print(f'Loaded metrics: {len(all_metrics)} rows across {len(runs_all)} runs')
print(all_metrics.groupby(['method','dataset','split']).size())


In [ ]:
import matplotlib.pyplot as plt

# Per-step train loss + rank-stability (только содержательные метрики)
KEEP_METRICS_PER_METHOD = {
    'adalora': ['energy_rank_95'],
    'l1ra':    ['energy_rank_95', 'active_rank_cal16', 'gate_mean'],
    'lora':    ['energy_rank_95'],
}

datasets = sorted(stab['dataset'].unique())
methods  = [m for m in ['adalora', 'l1ra', 'lora'] if m in stab['method'].unique()]

fig, axes = plt.subplots(len(datasets), len(methods),
                          figsize=(5*len(methods), 3.5*len(datasets)),
                          sharex=True, squeeze=False)

for row, ds in enumerate(datasets):
    for col, method in enumerate(methods):
        ax = axes[row][col]
        ax2 = ax.twinx()

        # Train loss (left axis)
        tl = all_metrics[(all_metrics['dataset'] == ds) &
                         (all_metrics['method']  == method) &
                         (all_metrics['split']   == 'train') &
                         (all_metrics['time_sec'].isna())]   # per-step строки
        if not tl.empty:
            ax.plot(tl['step'], tl['loss'], color='black', lw=0.7, alpha=0.4, label='train_loss')
            # smoothed
            if len(tl) > 20:
                sm = tl['loss'].rolling(50, min_periods=1).mean()
                ax.plot(tl['step'], sm, color='black', lw=1.5, label='train_loss (MA50)')

        # Rank-stability ρ (right axis)
        sub = stab[(stab['dataset']==ds) & (stab['method']==method)]
        for metric in KEEP_METRICS_PER_METHOD.get(method, []):
            g = sub[sub['metric']==metric].groupby('step')['rho_vs_final'].mean().reset_index()
            if not g.empty:
                ax2.plot(g['step'], g['rho_vs_final'], marker='o', ms=2.5, lw=1.2, label=metric)
        ax2.axhline(0.9, ls='--', color='gray', lw=0.7, alpha=0.6)

        # Vertical line at 30% of training
        if not tl.empty:
            step30 = int(tl['step'].max() * 0.3)
            ax.axvline(step30, color='red', ls=':', lw=1, alpha=0.7)
            ax.text(step30, ax.get_ylim()[1]*0.95, ' 30%',
                    color='red', fontsize=8, va='top')

        ax.set_title(f'{ds} — {method}')
        ax.set_xlabel('step')
        ax.set_ylabel('train_loss', color='black')
        ax2.set_ylabel('ρ(metric, final)', color='tab:blue')
        ax2.set_ylim(0, 1.05)
        ax.grid(alpha=0.25)
        # merged legend
        h1, l1 = ax.get_legend_handles_labels()
        h2, l2 = ax2.get_legend_handles_labels()
        ax.legend(h1+h2, l1+l2, fontsize=7, loc='center right')

plt.tight_layout()
plt.savefig(f'{OUT_DIR}/loss_vs_rank_stability.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved', f'{OUT_DIR}/loss_vs_rank_stability.png')


In [ ]:
# Cost vs quality summary — таблица для защиты
summary_rows = []

for (method, dataset, run_name), g in all_metrics.groupby(['method','dataset','run_name']):
    # эпоховые summary-строки (time_sec заполнен)
    eps = g[g['time_sec'].notna()]
    train_eps = eps[eps['split']=='train']
    val_eps   = eps[eps['split']=='val']
    if train_eps.empty or val_eps.empty:
        continue
    total_train_time = train_eps['time_sec'].sum()
    peak_gpu_mb = eps['gpu_mem_mb'].max()
    best_val_ppl = val_eps['ppl'].min()
    best_val_ep  = int(val_eps.loc[val_eps['ppl'].idxmin(), 'epoch'])
    final_val_ppl = val_eps.sort_values('step').iloc[-1]['ppl']

    summary_rows.append({
        'method': method, 'dataset': dataset, 'run_name': run_name,
        'total_train_sec': round(total_train_time, 1),
        'peak_gpu_gb': round(peak_gpu_mb/1024, 2),
        'best_val_ppl': round(best_val_ppl, 4),
        'best_val_epoch': best_val_ep,
        'final_val_ppl': round(final_val_ppl, 4),
    })

cost_summary = pd.DataFrame(summary_rows).sort_values(['dataset','method'])
cost_summary.to_csv(f'{OUT_DIR}/training_cost_summary.csv', index=False)
print(cost_summary.to_string(index=False))
print('\nSaved', f'{OUT_DIR}/training_cost_summary.csv')
